In [39]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import torch
import torch.nn as nn

In [41]:
# Preprocess new data
new_data_path = "C:/Users/Tharunraj_S/Desktop/Mini_project/Codes/test.csv"
df_new=pd.read_csv(new_data_path)
df_new.head()

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,4,51,80,1.01,2,0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [42]:
# dropping id column
df_new.drop('id', axis = 1, inplace = True)

In [43]:
df_new.drop('classification', axis = 1, inplace = True)

In [44]:
# rename column names to make it more user-friendly
df_new.columns = ['age', 'blood_pressure', 'specific_gravity', 'albumin', 'sugar', 'red_blood_cells', 'pus_cell',
              'pus_cell_clumps', 'bacteria', 'blood_glucose_random', 'blood_urea', 'serum_creatinine', 'sodium',
              'potassium', 'haemoglobin', 'packed_cell_volume', 'white_blood_cell_count', 'red_blood_cell_count',
              'hypertension', 'diabetes_mellitus', 'coronary_artery_disease', 'appetite', 'peda_edema',
              'aanemia']

In [45]:
# replace some incorrect values
df_new['diabetes_mellitus'].replace(to_replace = {'\tno':'no','\tyes':'yes',' yes':'yes'},inplace=True)

df_new['coronary_artery_disease'] = df_new['coronary_artery_disease'].replace(to_replace = '\tno', value='no')


In [46]:
# converting necessary columns to numerical type
df_new['packed_cell_volume'] = pd.to_numeric(df_new['packed_cell_volume'], errors='coerce')
df_new['white_blood_cell_count'] = pd.to_numeric(df_new['white_blood_cell_count'], errors='coerce')
df_new['red_blood_cell_count'] = pd.to_numeric(df_new['red_blood_cell_count'], errors='coerce')

In [47]:
# Extracting categorical and numerical columns
cat_cols = [col for col in df_new.columns if df_new[col].dtype == 'object']
num_cols = [col for col in df_new.columns if df_new[col].dtype != 'object']

In [48]:
df_new.head(10)

,age,blood_pressure,specific_gravity,albumin,sugar,red_blood_cells,pus_cell,pus_cell_clumps,bacteria,blood_glucose_random,...,haemoglobin,packed_cell_volume,white_blood_cell_count,red_blood_cell_count,hypertension,diabetes_mellitus,coronary_artery_disease,appetite,peda_edema,aanemia
0,51,80,1.01,2,0,normal,normal,notpresent,notpresent,106,...,11.6,35,7300,4.6,no,no,no,good,no,no


In [49]:
# filling null values, we will use two methods, random sampling for higher null values and mean/mode sampling for lower null values
def random_value_imputation(feature):
    random_sample = df_new[feature].dropna().sample(df_new[feature].isna().sum())
    random_sample.index = df_new[df_new[feature].isnull()].index
    df_new.loc[df_new[feature].isnull(), feature] = random_sample
    
def impute_mode(feature):
    mode = df_new[feature].mode()[0]
    df_new[feature] = df_new[feature].fillna(mode)

In [50]:
# filling "red_blood_cells" and "pus_cell" using random sampling method and rest of cat_cols using mode imputation
random_value_imputation('red_blood_cells')
random_value_imputation('pus_cell')

for col in cat_cols:
    impute_mode(col)

In [52]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in cat_cols:
    df_new[col] = le.fit_transform(df_new[col])

In [54]:
class KidneyNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(KidneyNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x
 

In [61]:
from sklearn.model_selection import train_test_split
import numpy as np
# Data normalization
X = df_new
y = ["ckd","notckd"]

# Convert Series to numpy array
X_test_array = X.to_numpy()

# Convert Series to numpy array and class mapping for detection
y_test_array = np.array(y)
class_mapping = {'ckd': 0, 'notckd': 1}
y_test_indices = [class_mapping[label] for label in y_test_array]

# Convert to PyTorch tensors
X_test_tensor = torch.tensor(X_test_array, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_indices, dtype=torch.long)


In [66]:

# Load the model architecture
loaded_model = KidneyNN(input_size, hidden_size, num_classes)
loaded_model.load_state_dict(torch.load(model_path))
loaded_model.eval()  # Set the model to evaluation mode

# Perform predictions
with torch.no_grad():
    outputs = loaded_model(X_test_tensor)
    
    
    _, predicted_classes = torch.max(outputs, 1)  # Get the predicted classes

# Convert tensor to Python list
predicted_classes_list = predicted_classes.tolist()
print('Predicted classes:', predicted_classes_list)


Predicted classes: [0]
